# 1.停滞エリア

* どこで時間を消費しているか？
* 農家や労働者がどこで最も時間を消費しているか？

#### 特徴量

`["farmer_x_y"]` 自分の座標<br>
`["hands_list_1"]` 労働者1座標<br>
.<br>
.<br>
`["hands_list_20"]` 労働者20座標<br>
`["unlocked_quadrants"]` アンロックした土地<br>

#### グラフ

* 2Dヒートマップ (2D Heatmap)
* ヒストグラム

**function file**:
`kaggriculture_df.py`
`kaggle_kaggriculture.py`
`my_plot.py`


#### このノートのメインURL

データセット：https://www.kaggle.com/datasets/nagatakengo/kaggriculture-movements-in-the-top-xx

メインnotebook：https://www.kaggle.com/code/nagatakengo/kaggriculture-movements-top-xx

## 2Dヒートマップ

In [ ]:



import sys
sys.path.append("/kaggle/input/datasets/nagatakengo/kaggriculture-movements-in-the-top-xx")



import pandas as pd
from kaggriculture_df import Kaggriculture_df
from my_plot import sns_Hist
import ast
from numpy import astype
from kaggle_kaggriculture import convert, xy_plt_2d, xy_plt_2d_all, create_all_hands_feature, new_xy_plt_2d
import matplotlib.pyplot as plt
import seaborn as sns


df_st = (Kaggriculture_df("/kaggle/input/datasets/nagatakengo/kaggriculture-movements-in-the-top-xx/results/st_data/strong_rank1_score157577.0_p1.json", 1)).df

"""
df_st = (Kaggriculture_df("/content/strong_rank1_score157577.0_p1.json", 1)).df

"""


In [ ]:
df_st

In [ ]:
xy_plt_2d(df_st["farmer_x_y"])

In [ ]:
xy_plt_2d_all(df_st, ["hands_list_1", "hands_list_2"], 1, 2)

In [ ]:
cols_st = ["farmer_x_y"] + [f"hands_list_{i}" for i in range(1, 17)]
xy_plt_2d_all(df_st, cols_st, 3, 6)

## 従業員達の総合的な動き

In [ ]:
df_st = create_all_hands_feature(df_st)

In [ ]:
new_xy_plt_2d(df_st["all_hands"])
xy_plt_2d(df_st["farmer_x_y"])

# 2.土地をアンロックしたタイミング

In [ ]:
df_st["unlocked_quadrants"].value_counts()

## ヒストグラム


* 土地をアンロックしたタイミング


In [ ]:
df_st_unlock = df_st[["step", "unlocked_quadrants"]].copy()
df_st_unlock["unlocked_quadrants"] = df_st_unlock["unlocked_quadrants"].astype(str)
sns_Hist(data=df_st_unlock, x="step", hue="unlocked_quadrants", figsize=(15,10), step=25, bins=720)

# 3.停滞エリアで起きている事

* その場所でなにが行われているか？

#### 特徴量

`["farmer_x_y"]` 農家の座標<br>
`["farmer_action"]`農家の行動<br><br>

`["hands_list_1"]` 労働者1座標<br>
.<br>
.<br>
`["hands_list_20"]` 労働者20座標<br>

`["hands_action_list_1"]` 労働者1行動<br>
.<br>
.<br>
`["hands_action_list_20"]` 労働者20行動<br>



#### グラフ

* 2Dヒートマップ (2D Heatmap)
* ヒストグラム


## 2Dヒートマップ

In [ ]:
#1
def workers_action(df):
    """
    df作成

    """
    base_cols = ["step", "farmer_x_y", "farmer_action"]
    hands_list_cols = [f"hands_list_{i}" for i in range(1, 15)]
    hands_action_cols = [f"hands_action_list_{i}" for i in range(1, 15)]



    return df[["money"] + base_cols + hands_list_cols + hands_action_cols].copy()

#2
def workers_count_list (df):
    """
    df全員分の行動を出力する
    """
    print(df_st_action["farmer_action"].value_counts())
    for i in range(1, 15):
        col_name = f"hands_action_list_{i}"
        if col_name in df.columns:
            print(f"--- {col_name}  ---")
            print(df[col_name].value_counts())
            print()

#4
def my_action_2d_plt(df_xy, df_action, target_action, figsize=(6, 5)):
    """
    特定のアクションが実行された座標をヒートマップに変換：単体

    <df_xy>: 2D配列のSeries (例: df["farmer_x_y"])
    <df_action>: アクションのSeries (例: df["farmer_action"])
    <target_action>: 可視化したいアクションの文字列 (例: "WATER" や "FEED, WHEAT")
    <figsize>: グラフのサイズ設定
    """
    df = pd.DataFrame({
        "x_y": df_xy,
        "action": df_action
    })

    df_filtered = df[df["action"].astype(str).str.contains(target_action, na=False, regex=False)]

    df_filtered_xy = df_filtered["x_y"].apply(convert)
    df_plot = pd.DataFrame(df_filtered_xy.tolist(), columns=["x", "y"])
    df_plot = df_plot.dropna().astype(int)

    heatmap_data = pd.crosstab(df_plot['y'], df_plot['x'])

    grid_range = list(range(10))
    heatmap_data = heatmap_data.reindex(
        index=grid_range, columns=grid_range, fill_value=0
    )

    plt.figure(figsize=figsize)
    sns.heatmap(heatmap_data, annot=True, cmap="Blues", fmt="d")
    plt.title(f"Action: {target_action}")
    plt.show()
#5
def my_action_2d_plt_all(df_xy, df_action, target_actions, n_rows, n_cols):
    """
    複数のアクションをそれぞれのヒートマップに変換し、一気に描画する

    <df_xy>: 2D配列のSeries (例: df["farmer_x_y"])
    <df_action>: アクションのSeries (例: df["farmer_action"])
    <target_actions>: 可視化したいアクションのリスト (例: ["PLANT", "WATER", "HARVEST"])
    <n_rows>: 行の数
    <n_cols>: 列の数
    """
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))

    if n_rows * n_cols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    grid_range = list(range(10))

    df = pd.DataFrame({
        "x_y": df_xy,
        "action": df_action
    })

    for i, action_str in enumerate(target_actions):
        ax = axes[i]

        df_filtered = df[df["action"].astype(str).str.contains(action_str, na=False, regex=False)]

        df_filtered_xy = df_filtered["x_y"].apply(convert)
        df_plot = pd.DataFrame(df_filtered_xy.tolist(), columns=["x", "y"])
        df_plot = df_plot.dropna().astype(int)

        if not df_plot.empty:
            heatmap_data = pd.crosstab(df_plot['y'], df_plot['x'])
            heatmap_data = heatmap_data.reindex(
                index=grid_range, columns=grid_range, fill_value=0
            )
        else:
            heatmap_data = pd.DataFrame(0, index=grid_range, columns=grid_range)
        ax.grid(False)
        sns.heatmap(heatmap_data, annot=True, cmap="Blues", fmt="d", ax=ax, cbar=False)
        ax.set_title(f"Action: {action_str}")

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()


df_st_action = workers_action(df_st)

In [ ]:
df_st_action

In [ ]:
df_st_action.columns

## そこで何が行われているかの特徴量の考察
| **アクションカテゴリ** | **実行可能なコマンド**                                                                  | **備考**                                                |
| ------------- | ------------------------------------------------------------------------------ | ------------------------------------------------------------ |
| **移動・運搬**     | `PICKUP`:小屋（Shed）にあるアイテム（動物、肥料、収穫物など）を最大N個、手持ち（インベントリ）に取り出します。種は専用スロットにあるため対象外です<br><br> `DROP`:手持ちのアイテムをマスに置きます。小屋に隣接している場合は小屋の中に格納します。<br><br> `PLACE`:手持ちの動物を対応する空の施設（牧草地など）に配置します。<br><br>                    | |
| **農業（植物）**    | `PLANT`:購入して共有スロットにある「種」を現在のマスに植えます。<br><br> `WATER`:植物に水をやります。1日1回のみ有効です<br><br> `HARVEST`: 成長した植物から農産物を収穫し、手持ちに追加します。<br><br>`FERTILIZE`: 植物に肥料を与えます。その後の3日間、水やり時のボーナス収穫量が倍増します。<br><br>`DIG`: マスに発生した雑草（WEED）を取り除きます。<br><br>                                | |
| **畜産（動物）**    | `BUILD_COOP`: 鳥小屋（Gooseなどの鳥用）を建設します。<br><br>  `BUILD_PASTURE`: 牧草地（牛や羊用）を建設します。<br><br> `FEED`: 手持ちの作物（小麦など）を消費して、動物にエサを与えます（1日1回）<br><br> `COLLECT_FERTILIZER`:動物から排出された肥料（FERTILIZER）を回収します。動物1頭につき1日1つ生成されます。<br><br> `CARE`:動物の世話をします（1日1回）<br><br> |  |
| 市場取引      | `BUY_SEED`:市場から指定した「種」を購入します。<br><br> `BUY_PRODUCT`:市場から農産物や肥料などのアイテムを購入します。<br><br> `BUY_ANIMAL`: 市場から動物を購入します。<br><br>`SELL`: 手持ち、または小屋にあるアイテムを市場に売却します。種は売却できません。<br><br>                                |                        |
| 雇用と拡張     | `HIRE`:その日限定の労働者（Farm hand）を雇います。同日に雇う人数が増えるほど採用コストが上がります。<br><br> `BUY_LAND`: 隣接する新しい5x5の土地（区画）を購入し、農場エリアを拡張します。<br><br>                                                             |                   |

<br><br><br><br>  


<br><br>

**従業員の足跡**


## 移動・運搬

```
PICKUP:小屋（Shed）にあるアイテム（動物、肥料、収穫物など）を最大N個、手持ち（インベントリ）に取り出します。種は専用スロットにあるため対象外です

DROP:手持ちのアイテムをマスに置きます。小屋に隣接している場合は小屋の中に格納します。

PLACE:手持ちの動物を対応する空の施設（牧草地など）に配置します。
```

* 農家

In [ ]:
target_list = ["PICKUP", "DROP", "PLACE"]
my_action_2d_plt_all(df_st_action["farmer_x_y"], df_st_action["farmer_action"], target_list, 1, 3)

* 従業員

In [ ]:
def ac_move(num):
    print(f"従業員{num}")
    target_list = ["PICKUP", "DROP", "PLACE"]
    my_action_2d_plt_all(df_st[f'hands_list_{num}'], df_st[f'hands_action_list_{num}'], target_list, 1, 3)

for i in range(1, 15):
    ac_move(i)

## 農業（植物）

```
PLANT:購入して共有スロットにある「種」を現在のマスに植えます。

WATER:植物に水をやります。1日1回のみ有効です

HARVEST: 成長した植物から農産物を収穫し、手持ちに追加します。

FERTILIZE: 植物に肥料を与えます。その後の3日間、水やり時のボーナス収穫量が倍増します。

DIG: マスに発生した雑草（WEED）を取り除きます。
```

* 農家

In [ ]:
target_list = ["PLANT", "WATER", "HARVEST","FERTILIZE","DIG" ]
my_action_2d_plt_all(df_st_action["farmer_x_y"], df_st_action["farmer_action"], target_list, 2, 3)

In [ ]:
def ac_agriculture(num):
    print(f"従業員{num}")
    target_list = ["PLANT", "WATER", "HARVEST","FERTILIZE","DIG" ]
    my_action_2d_plt_all(df_st[f'hands_list_{num}'], df_st[f'hands_action_list_{num}'], target_list, 2, 3)

for i in range(1, 15):
    ac_agriculture(i)

## 畜産（動物）

```
BUILD_COOP: 鳥小屋（Gooseなどの鳥用）を建設します。

BUILD_PASTURE: 牧草地（牛や羊用）を建設します。

FEED: 手持ちの作物（小麦など）を消費して、動物にエサを与えます（1日1回）

COLLECT_FERTILIZER:動物から排出された肥料（FERTILIZER）を回収します。動物1頭につき1日1つ生成されます。

CARE:動物の世話をします（1日1回）
```

* 農家

In [ ]:
target_list = ["BUILD_COOP", "BUILD_PASTURE", "FEED","COLLECT_FERTILIZER","CARE" ]
my_action_2d_plt_all(df_st_action["farmer_x_y"], df_st_action["farmer_action"], target_list, 2, 3)

In [ ]:
def ac_animal(num):
    print(f"従業員{num}")
    target_list = ["BUILD_COOP", "BUILD_PASTURE", "FEED","COLLECT_FERTILIZER","CARE" ]
    my_action_2d_plt_all(df_st[f'hands_list_{num}'], df_st[f'hands_action_list_{num}'], target_list, 2, 3)

for i in range(1, 15):
    ac_animal(i)

# 4.それぞれの役割と足跡まとめ

|||
|---|---|
|NW|NE|
|SW|SE|

## アニメーションプロット

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import numpy as np

def route_animation(data):
    xy = np.array(data.tolist(), dtype=float)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-0.5, 9.5)
    ax.set_ylim(9.5, -0.5)
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    ax.set_aspect("equal")
    ax.grid()

    line, = ax.plot([], [], alpha=.5)
    point, = ax.plot([], [], "ro")

    def update(i):
        line.set_data(xy[:i, 0], xy[:i, 1])
        point.set_data([xy[i-1, 0]], [xy[i-1, 1]])
        return line, point

    ani = FuncAnimation(
        fig,
        update,
        frames=range(1, len(xy) + 1, 5),
        interval=250
    )

    plt.close(fig)
    return HTML(ani.to_jshtml())

## 農家


|行動|種目|コマンド|
|-----|-----|-----|
|肥料をまく|農業(植物)|`FERTILIZE`|
|餌やり|畜産（動物）|`FEED`|
|肥料回収|畜産（動物）|`COLLECT_FERTILIZER`|
|動物の世話|畜産（動物）|`CARE`|

In [ ]:
display(route_animation(df_st["farmer_x_y"]))

## 従業員1

|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|餌やり|畜産（動物）|`FEED`|
|肥料回収|畜産（動物）|`COLLECT_FERTILIZER`|
|肥料をまく||`FERTILIZE`|

In [ ]:
display(route_animation(df_st["hands_list_1"]))

## 従業員2
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|植物回収|農業(植物)|`HARVEST`|
|肥料をまく|農業(植物)|`FERTILIZE`|
|餌やり|畜産（動物）|`FEED`|
|肥料回収|畜産（動物）|`COLLECT_FERTILIZER`|
|動物の世話|畜産（動物）|`CARE`|

In [ ]:
display(route_animation(df_st["hands_list_2"]))

## 従業員3
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|餌やり|畜産（動物）|`FEED`|
|肥料回収|畜産（動物）|`COLLECT_FERTILIZER`|
|動物の世話|畜産（動物）|`CARE`|
|植物回収|農業(植物)|`HARVEST`|

In [ ]:
display(route_animation(df_st["hands_list_3"]))

## 従業員4
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|種を植える|農業(植物)|`PLANT`|
|植物回収|農業(植物)|`HARVEST`|
|牧草地建設|畜産（動物）|`BUILD_PASTURE`|
|肥料をまく|農業(植物)|`FERTILIZE`|
|動物の世話|畜産（動物）|`CARE`|
|餌やり|畜産（動物）|`FEED`|
|肥料回収|畜産（動物）|`COLLECT_FERTILIZER`|

In [ ]:
display(route_animation(df_st["hands_list_4"]))

## 従業員5
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|植物回収|農業(植物)|`HARVEST`|

In [ ]:
display(route_animation(df_st["hands_list_5"]))

## 従業員6
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|

In [ ]:
display(route_animation(df_st["hands_list_6"]))

## 従業員7
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|

In [ ]:
display(route_animation(df_st["hands_list_7"]))

## 従業員8
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|植物回収|農業(植物)|`HARVEST`|

In [ ]:
display(route_animation(df_st["hands_list_8"]))

## 従業員9
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|

In [ ]:
display(route_animation(df_st["hands_list_9"]))

## 従業員10
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|植物回収|農業(植物)|`HARVEST`|
|種を植える|農業(植物)|`PLANT`|

In [ ]:
display(route_animation(df_st["hands_list_10"]))

## 従業員11
|行動|種目|コマンド|
|-----|-----|-----|
|種を植える|農業(植物)|`PLANT`|
|植物回収|農業(植物)|`HARVEST`|
|水やり|農業(植物)|`WATER`|

In [ ]:
display(route_animation(df_st["hands_list_11"]))

## 従業員12
|行動|種目|コマンド|
|-----|-----|-----|
|植物回収|農業(植物)|`HARVEST`|

In [ ]:
display(route_animation(df_st["hands_list_12"]))

## 従業員13
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|肥料をまく|農業(植物)|`FERTILIZE`|

In [ ]:
display(route_animation(df_st["hands_list_13"]))

## 従業員14
|行動|種目|コマンド|
|-----|-----|-----|
|水やり|農業(植物)|`WATER`|
|肥料をまく|農業(植物)|`FERTILIZE`|

In [ ]:
display(route_animation(df_st["hands_list_14"]))